In [1]:
import json
import asyncio
import sys
import sqlite3

from pathlib import Path

# Walk up from CWD to find the project root (identified by pyproject.toml),
# so imports work regardless of where Jupyter is launched from.
def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(f"Could not find project root from {start}")

project_root = find_project_root(Path().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from movie_ingestion.metadata_generation.inputs import MovieInputData
from movie_ingestion.metadata_generation.generators.plot_events import generate_plot_events
from movie_ingestion.metadata_generation.generators.reception import generate_reception
from movie_ingestion.metadata_generation.generators.plot_analysis import generate_plot_analysis
from movie_ingestion.metadata_generation.generators.viewer_experience import generate_viewer_experience
from movie_ingestion.metadata_generation.generators.watch_context import generate_watch_context
from movie_ingestion.metadata_generation.generators.narrative_techniques import generate_narrative_techniques
from movie_ingestion.metadata_generation.generators.production_keywords import generate_production_keywords
from movie_ingestion.metadata_generation.generators.source_of_inspiration import generate_source_of_inspiration
from implementation.llms.generic_methods import LLMProvider
from movie_ingestion.tracker import deserialize_imdb_row

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ORIGINAL_SET_TMDB_IDS = [9377, 269149, 1584, 109445, 2493, 354912, 508965, 14160, 10674, 808, 13397, 76341, 85, 155, 245891, 1771, 569094, 299534, 11, 671, 120, 98, 27205, 603, 157336, 335984, 329, 329865, 493922, 694, 49018, 1034541, 176, 807, 496243, 419430, 1359, 550, 597, 13, 666277, 423, 11036, 1824, 25195, 216015, 392044, 545611, 22538, 37136]
MEDIUM_SPARSITY_TMDB_IDS = [329974, 1498832, 821937, 92, 160, 45739, 576560, 1383243, 1642210, 1639488]
HIGH_SPARSITY_TMDB_IDS = [270909, 493103, 64262, 1611977, 706910, 1297426, 35952, 158227, 215782, 1642486]

ALL_TMDB_IDS = ORIGINAL_SET_TMDB_IDS + MEDIUM_SPARSITY_TMDB_IDS + HIGH_SPARSITY_TMDB_IDS

# Open the tracker DB directly using the project root from the setup cell,
# avoiding reliance on CWD (init_db() uses a relative path).
db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
db.row_factory = sqlite3.Row

placeholders = ",".join("?" * len(ALL_TMDB_IDS))

# Fetch title + release_date from tmdb_data (the only fields tmdb_data has
# that we need; overview text lives in imdb_data.overview, not tmdb_data).
tmdb_rows: dict[int, sqlite3.Row] = {
    row["tmdb_id"]: row
    for row in db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_TMDB_IDS,
    ).fetchall()
}

# Fetch all imdb_data columns; deserialize_imdb_row parses JSON array columns.
imdb_rows: dict[int, dict] = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_TMDB_IDS,
    ).fetchall()
}

db.close()


def build_movie_input(tmdb_id: int) -> MovieInputData | None:
    tmdb = tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = imdb_rows.get(tmdb_id, {})

    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None

    # Prefer IMDB maturity_rating; fall back to TMDB where IMDB has none.
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""

    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],      # imdb_data column is "synopses"
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],              # not stored in the DB
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )


movies = [m for tmdb_id in ALL_TMDB_IDS if (m := build_movie_input(tmdb_id)) is not None]
print(f"Built {len(movies)} MovieInputData objects from {len(ALL_TMDB_IDS)} requested IDs")

Built 70 MovieInputData objects from 70 requested IDs


In [3]:
from dataclasses import dataclass, field as dc_field


@dataclass(frozen=True)
class PlaygroundCandidate:
    """A model configuration to test in the playground.

    Supports optional system_prompt and response_format overrides for
    experiments that need non-default prompts or schemas (e.g. the
    with-justifications variant). When not set, generator functions
    use their own internal defaults.
    """
    label: str
    provider: LLMProvider
    model: str
    kwargs: dict = dc_field(default_factory=dict)
    system_prompt: str | None = None
    response_format: type | None = None


CANDIDATES = [
    PlaygroundCandidate(
        "gpt-5-mini-minimal", LLMProvider.OPENAI, "gpt-5-mini",
        {"reasoning_effort": "minimal", "verbosity": "low"},
    ),
    # PlaygroundCandidate(
    #     "gpt-5-mini-low-new-prompt", LLMProvider.OPENAI, "gpt-5-mini",
    #     {"reasoning_effort": "low", "verbosity": "low"},
    # ),
    # PlaygroundCandidate(
    #     "gpt-5-mini-medium-new-prompt", LLMProvider.OPENAI, "gpt-5-mini",
    #     {"reasoning_effort": "medium", "verbosity": "low"},
    # ),
    # PlaygroundCandidate(
    #     "gpt-5.4-nano-medium", LLMProvider.OPENAI, "gpt-5.4-nano",
    #     {"reasoning_effort": "medium", "verbosity": "low"},
    # ),
    # PlaygroundCandidate(
    #     "gpt-5-nano", LLMProvider.OPENAI, "gpt-5-nano",
    #     {"reasoning_effort": "low", "verbosity": "low"},
    # ),
    # PlaygroundCandidate(
    #     "kimi-k2.5-no-thinking", LLMProvider.KIMI, "kimi-k2.5",
    #     # Thinking disabled — closest Kimi equivalent to gpt-5-mini reasoning_effort="low";
    #     # Kimi has no granular thinking budget, only on/off via enable_thinking
    #     {"enable_thinking": False},
    # ),
    # PlaygroundCandidate(
    #     "qwen3.5-flash", LLMProvider.ALIBABA, "qwen3.5-flash",
    #     # enable_thinking must be False — structured output not supported in thinking mode
    #     {"temperature": 0.2, "extra_body": {"enable_thinking": False}},
    # ),
    # PlaygroundCandidate(
    #     "gemini-2.5-flash-low", LLMProvider.GEMINI, "gemini-2.5-flash",
    #     # Small thinking budget: task involves cross-review interpretation, not standardized extraction
    #     {"temperature": 0.2, "thinking_config": {"thinking_budget": 1024}},
    # ),
    # PlaygroundCandidate(
    #     "gemini-2.5-flash-medium", LLMProvider.GEMINI, "gemini-2.5-flash",
    #     # Small thinking budget: task involves cross-review interpretation, not standardized extraction
    #     {"temperature": 0.2, "thinking_config": {"thinking_budget": 2048}},
    # ),
    # PlaygroundCandidate(
    #     "gemini-2.5-flash-lite", LLMProvider.GEMINI, "gemini-2.5-flash-lite",
    #     # Flash Lite does not support thinking configuration
    #     {"temperature": 0.2},
    # ),
    # PlaygroundCandidate(
    #     "gpt-oss-120b-high", LLMProvider.GROQ, "openai/gpt-oss-120b",
    #     {"temperature": 0.2, "reasoning_effort": "high", "include_reasoning": False},
    # ),
    # PlaygroundCandidate(
    #     "llama-4-scout", LLMProvider.GROQ, "meta-llama/llama-4-scout-17b-16e-instruct",
    #     {"temperature": 0.2},
    # ),
]

print(f"Defined {len(CANDIDATES)} candidates")

Defined 1 candidates


In [4]:
from pydantic import BaseModel


def print_all_fields(result: BaseModel) -> None:
    """Print every field of a Pydantic result on its own line.

    Solves the problem where __str__() omits certain fields
    (e.g., ReceptionOutput omits review_insights_brief).
    """
    for field_name in type(result).model_fields:
        value = getattr(result, field_name)
        if isinstance(value, list):
            print(f"  {field_name}:")
            for item in value:
                print(f"    - {item}")
        elif isinstance(value, BaseModel):
            print(f"  {field_name}:")
            for sub_field in type(value).model_fields:
                sub_value = getattr(value, sub_field)
                print(f"    {sub_field}: {sub_value}")
        else:
            print(f"  {field_name}: {value}")


async def run_candidates(movie: MovieInputData, generator_fn, extra_kwargs: dict | None = None) -> None:
    """Run all candidates through a generator function, printing results and token usage.

    Args:
        movie: The movie input data to test with.
        generator_fn: An async generator function (e.g., generate_reception).
        extra_kwargs: Additional keyword args passed to the generator before
            candidate kwargs (e.g., plot_summary, review_insights_brief).
    """
    if extra_kwargs is None:
        extra_kwargs = {}

    for candidate in CANDIDATES:
        print(f"\n{'─' * 60}")
        print(f"Model: {candidate.label}")
        try:
            result, token_usage = await generator_fn(
                movie,
                **extra_kwargs,
                provider=candidate.provider,
                model=candidate.model,
                **candidate.kwargs,
            )
            print_all_fields(result)
            print(f"  token_usage: {token_usage}")
        except Exception as e:
            print(f"  ERROR: {e}")

In [5]:
MODEL_PRICING: dict[str, tuple[float, float]] = {
    # (input_price_per_M, output_price_per_M)
    "qwen3.5-flash":                                   (0.10, 0.40),
    "gemini-2.5-flash":                                (0.30, 2.50),
    "gemini-2.5-flash-lite":                           (0.10, 0.40),
    "gpt-5-mini":                                      (0.25, 2.00),
    "gpt-5-nano":                                      (0.05, 0.40),
    "gpt-5.4-nano":                                    (0.20, 1.25),
    "openai/gpt-oss-120b":                             (0.15, 0.60),
    "meta-llama/llama-4-scout-17b-16e-instruct":       (0.11, 0.34),
    "kimi-k2.5":                                       (0.60, 3.00),
    "gemini-3.1-flash-lite-preview":                   (0.25, 1.50),
}

In [6]:
# ── Reception (Wave 1) ───────────────────────────────────────────────────────
# Inputs: reception_summary, audience_reception_attributes, featured_reviews
# Buckets: 6 groups by combined_top5_chars (review richness)

import json as _json
from movie_ingestion.metadata_generation.generators.reception import (
    build_reception_user_prompt,
    generate_reception,
)

# ── Load bucket IDs from reception_eval_buckets.json ─────────────────────────
_buckets_path = project_root / "ingestion_data" / "reception_eval_buckets.json"
with open(_buckets_path) as _f:
    _buckets = _json.load(_f)

ULTRA_THIN_0_1K = [s["tmdb_id"] for s in _buckets["ultra_thin_0_1000"]["samples"]]
VERY_THIN_1K_2500 = [s["tmdb_id"] for s in _buckets["very_thin_1000_2500"]["samples"]]
THIN_2500_5K = [s["tmdb_id"] for s in _buckets["thin_2500_5000"]["samples"]]
MODERATE_5K_7500 = [s["tmdb_id"] for s in _buckets["moderate_5000_7500"]["samples"]]
RICH_7500_10500 = [s["tmdb_id"] for s in _buckets["rich_7500_10500"]["samples"]]
VERY_RICH_10500_PLUS = [s["tmdb_id"] for s in _buckets["very_rich_10500_plus"]["samples"]]

EVAL_GROUPS = {
    "ultra_thin_0_1k": ULTRA_THIN_0_1K,
    "very_thin_1k_2500": VERY_THIN_1K_2500,
    "thin_2500_5k": THIN_2500_5K,
    "moderate_5k_7500": MODERATE_5K_7500,
    "rich_7500_10500": RICH_7500_10500,
    "very_rich_10500_plus": VERY_RICH_10500_PLUS,
}

# Load MovieInputData for all evaluation IDs
ALL_EVAL_IDS = [tid for group in EVAL_GROUPS.values() for tid in group]

db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
db.row_factory = sqlite3.Row

placeholders = ",".join("?" * len(ALL_EVAL_IDS))
eval_tmdb_rows = {
    row["tmdb_id"]: row
    for row in db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_EVAL_IDS,
    ).fetchall()
}
eval_imdb_rows = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_EVAL_IDS,
    ).fetchall()
}
db.close()

def _build_eval_movie(tmdb_id: int) -> MovieInputData | None:
    tmdb = eval_tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = eval_imdb_rows.get(tmdb_id, {})
    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )

eval_movies: dict[int, MovieInputData] = {}
for tid in ALL_EVAL_IDS:
    m = _build_eval_movie(tid)
    if m is not None:
        eval_movies[tid] = m
print(f"Built {len(eval_movies)} MovieInputData objects for evaluation")

# ── Use first 4 candidates ──────────────────────────────────────────────────
reception_candidates = CANDIDATES
print(f"Using {len(reception_candidates)} candidates:")
for c in reception_candidates:
    print(f"  - {c.label} ({c.model})")


def _compute_cost(model: str, input_tokens: int, output_tokens: int) -> float | None:
    """Compute generation cost in USD from MODEL_PRICING."""
    pricing = MODEL_PRICING.get(model)
    if pricing is None:
        return None
    input_price, output_price = pricing
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


async def _generate_for_candidate(
    movie: MovieInputData,
    candidate: "PlaygroundCandidate",
) -> dict:
    """Generate reception for one (movie, candidate) pair. Returns a result dict."""
    try:
        result, token_usage = await generate_reception(
            movie,
            provider=candidate.provider,
            model=candidate.model,
            **candidate.kwargs,
        )
        cost = _compute_cost(
            token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
        )
        cost_str = f"${cost:.6f}" if cost is not None else "n/a"
        print(
            f"  ✓ {candidate.label}: "
            f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
            f"cost={cost_str}"
        )
        return {
            "candidate": candidate.label,
            "model": candidate.model,
            "result": result.model_dump(),
            "input_tokens": token_usage.input_tokens,
            "output_tokens": token_usage.output_tokens,
            "cost_usd": cost,
        }
    except Exception as e:
        print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
        return {
            "candidate": candidate.label,
            "model": candidate.model,
            "result": None,
            "error": f"{type(e).__name__}: {e}",
            "input_tokens": None,
            "output_tokens": None,
            "cost_usd": None,
        }


def _merge_candidate_results(
    existing_results: list[dict],
    new_results: list[dict],
) -> list[dict]:
    """Merge new candidate results into existing ones.

    If a candidate with the same label already exists, replace that entry.
    Otherwise append the new entry. Preserves ordering of existing entries.
    """
    # Index existing results by label for O(1) lookup
    merged = {r["candidate"]: r for r in existing_results}
    # Overwrite or insert new results
    for r in new_results:
        merged[r["candidate"]] = r
    # Preserve original order for existing entries, append new ones at end
    seen = set()
    ordered = []
    for r in existing_results:
        label = r["candidate"]
        ordered.append(merged[label])
        seen.add(label)
    for r in new_results:
        if r["candidate"] not in seen:
            ordered.append(merged[r["candidate"]])
            seen.add(r["candidate"])
    return ordered

Built 36 MovieInputData objects for evaluation
Using 1 candidates:
  - gpt-5-mini-minimal (gpt-5-mini)


In [7]:
# # ── Generate per-movie, all candidates concurrently ─────────────────────────

# eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
# eval_dir.mkdir(parents=True, exist_ok=True)

# total_movies = len(eval_movies)
# for movie_idx, (tmdb_id, movie) in enumerate(eval_movies.items(), 1):
#     title = movie.title_with_year()
#     user_prompt = build_reception_user_prompt(movie)

#     print(f"\n[{movie_idx}/{total_movies}] {title} (tmdb_id={tmdb_id})")

#     # Fire all candidates concurrently — each hits a different provider,
#     # so concurrent calls spread rate-limit pressure across providers.
#     candidate_results = await asyncio.gather(*[
#         _generate_for_candidate(movie, c) for c in reception_candidates
#     ])

#     # Load existing file (if any) and merge new results into it
#     out_path = eval_dir / f"reception_{tmdb_id}.json"
#     if out_path.exists():
#         existing_output = _json.loads(out_path.read_text())
#         existing_candidates = existing_output.get("candidate_results", [])
#         merged = _merge_candidate_results(existing_candidates, list(candidate_results))
#     else:
#         merged = list(candidate_results)

#     movie_output = {
#         "tmdb_id": tmdb_id,
#         "title": title,
#         "user_prompt": user_prompt,
#         "candidate_results": merged,
#     }
#     out_path.write_text(_json.dumps(movie_output, indent=2, ensure_ascii=False))
#     print(f"  Saved → {out_path.name} ({len(merged)} candidates)")

# print(f"\nDone: {total_movies} movies × {len(reception_candidates)} candidates")

In [8]:
# # ── Plot Analysis Experiment 2: justification schema impact ──────────────────
# # Tests whether the with-justifications schema/prompt changes plot_analysis
# # quality compared to the standard schema, across 8 buckets.
# #
# # For each movie: 3 candidates = 3 runs.
# # All candidates omit emotional_observations and use the justification schema.
# # Results saved per-movie to evaluation_data/plot_analysis_{tmdb_id}.json.

# import json as _json
# import asyncio
# from movie_ingestion.metadata_generation.generators.plot_analysis import (
#     build_plot_analysis_user_prompt,
#     generate_plot_analysis,
# )
# from movie_ingestion.metadata_generation.schemas import (
#     PlotAnalysisOutput,
#     PlotAnalysisWithJustificationsOutput,
# )
# from movie_ingestion.metadata_generation.prompts.plot_analysis import (
#     SYSTEM_PROMPT as PA_SYSTEM_PROMPT,
#     SYSTEM_PROMPT_WITH_JUSTIFICATIONS as PA_SYSTEM_PROMPT_JUSTIFICATIONS,
# )

# # ── Candidates: 3 models, all justification schema ──────────────────────────

# @dataclass(frozen=True)
# class _PACandidate:
#     """Extends PlaygroundCandidate with schema/prompt pairing."""
#     label: str
#     provider: LLMProvider
#     model: str
#     system_prompt: str
#     response_format: type
#     kwargs: dict = dc_field(default_factory=dict)

# PLOT_ANALYSIS_CANDIDATES = [
#     _PACandidate(
#         "kimi-k2.5-no-thinking", LLMProvider.KIMI, "kimi-k2.5",
#         PA_SYSTEM_PROMPT_JUSTIFICATIONS, PlotAnalysisWithJustificationsOutput,
#         {"enable_thinking": False},
#     ),
#     _PACandidate(
#         "qwen3.5-flash-no-thinking", LLMProvider.ALIBABA, "qwen3.5-flash",
#         PA_SYSTEM_PROMPT_JUSTIFICATIONS, PlotAnalysisWithJustificationsOutput,
#         {"temperature": 0.1, "extra_body": {"enable_thinking": False}},
#     ),
#     _PACandidate(
#         "gpt-5.4-nano-medium", LLMProvider.OPENAI, "gpt-5.4-nano",
#         PA_SYSTEM_PROMPT_JUSTIFICATIONS, PlotAnalysisWithJustificationsOutput,
#         {"reasoning_effort": "medium", "verbosity": "low"},
#     ),
# ]

# # ── Load buckets and build MovieInputData ────────────────────────────────────

# _pa_buckets_path = project_root / "ingestion_data" / "plot_analysis_eval_buckets.json"
# with open(_pa_buckets_path) as _f:
#     _pa_config = _json.load(_f)

# # Collect all unique tmdb_ids across all buckets
# _pa_all_ids = list({
#     tid
#     for bucket in _pa_config["buckets"].values()
#     for tid in bucket["tmdb_ids"]
# })

# _pa_db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
# _pa_db.row_factory = sqlite3.Row

# _pa_placeholders = ",".join("?" * len(_pa_all_ids))

# # Load tmdb + imdb rows
# _pa_tmdb_rows = {
#     row["tmdb_id"]: row
#     for row in _pa_db.execute(
#         f"SELECT tmdb_id, title, release_date, maturity_rating "
#         f"FROM tmdb_data WHERE tmdb_id IN ({_pa_placeholders})",
#         _pa_all_ids,
#     ).fetchall()
# }
# _pa_imdb_rows = {
#     row["tmdb_id"]: deserialize_imdb_row(row)
#     for row in _pa_db.execute(
#         f"SELECT * FROM imdb_data WHERE tmdb_id IN ({_pa_placeholders})",
#         _pa_all_ids,
#     ).fetchall()
# }

# # Load Wave 1 outputs (plot_events + reception) from generated_metadata
# _pa_wave1 = {}
# for row in _pa_db.execute(
#     f"SELECT tmdb_id, plot_events, reception "
#     f"FROM generated_metadata WHERE tmdb_id IN ({_pa_placeholders})",
#     _pa_all_ids,
# ).fetchall():
#     _pa_wave1[row["tmdb_id"]] = {
#         "plot_events": _json.loads(row["plot_events"]) if row["plot_events"] else None,
#         "reception": _json.loads(row["reception"]) if row["reception"] else None,
#     }

# _pa_db.close()


# def _build_pa_movie(tmdb_id: int) -> MovieInputData | None:
#     tmdb = _pa_tmdb_rows.get(tmdb_id)
#     if tmdb is None:
#         print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
#         return None
#     imdb = _pa_imdb_rows.get(tmdb_id, {})
#     release_date = tmdb["release_date"] or ""
#     release_year = int(release_date[:4]) if release_date else None
#     maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
#     return MovieInputData(
#         tmdb_id=tmdb_id,
#         title=tmdb["title"],
#         release_year=release_year,
#         overview=imdb.get("overview") or "",
#         genres=imdb.get("genres") or [],
#         plot_synopses=imdb.get("synopses") or [],
#         plot_summaries=imdb.get("plot_summaries") or [],
#         plot_keywords=imdb.get("plot_keywords") or [],
#         overall_keywords=imdb.get("overall_keywords") or [],
#         featured_reviews=imdb.get("featured_reviews") or [],
#         reception_summary=imdb.get("reception_summary"),
#         audience_reception_attributes=imdb.get("review_themes") or [],
#         maturity_rating=maturity_rating,
#         maturity_reasoning=imdb.get("maturity_reasoning") or [],
#         parental_guide_items=imdb.get("parental_guide_items") or [],
#     )


# # Build MovieInputData + extract Wave 1 intermediates for each movie
# _pa_movies: dict[int, dict] = {}
# for tid in _pa_all_ids:
#     movie = _build_pa_movie(tid)
#     if movie is None:
#         continue
#     w1 = _pa_wave1.get(tid, {})
#     pe = w1.get("plot_events")
#     rec = w1.get("reception")
#     _pa_movies[tid] = {
#         "movie": movie,
#         "plot_synopsis": pe["plot_summary"] if pe else None,
#         "thematic_observations": rec.get("thematic_observations") if rec else None,
#     }

# print(f"Built {len(_pa_movies)} movies for plot_analysis evaluation")
# print(f"Buckets: {list(_pa_config['buckets'].keys())}")
# print(f"Candidates: {[c.label for c in PLOT_ANALYSIS_CANDIDATES]}")


# # ── Generation helpers ───────────────────────────────────────────────────────

# _PA_CONCURRENCY = asyncio.Semaphore(6)


# async def _pa_generate_one(
#     movie: MovieInputData,
#     plot_synopsis: str | None,
#     thematic_observations: str | None,
#     candidate: _PACandidate,
# ) -> dict:
#     """Generate plot_analysis for one (movie, candidate) combo."""
#     async with _PA_CONCURRENCY:
#         try:
#             result, token_usage = await generate_plot_analysis(
#                 movie,
#                 plot_synopsis=plot_synopsis,
#                 thematic_observations=thematic_observations,
#                 emotional_observations=None,
#                 provider=candidate.provider,
#                 model=candidate.model,
#                 system_prompt=candidate.system_prompt,
#                 response_format=candidate.response_format,
#                 **candidate.kwargs,
#             )
#             cost = _compute_cost(
#                 token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
#             )
#             return {
#                 "candidate": candidate.label,
#                 "label": candidate.label,
#                 "model": candidate.model,
#                 "result": result.model_dump(),
#                 "input_tokens": token_usage.input_tokens,
#                 "output_tokens": token_usage.output_tokens,
#                 "cost_usd": cost,
#             }
#         except Exception as e:
#             print(f"  ✗ {candidate.label} for {movie.title_with_year()}: {type(e).__name__}: {e}")
#             return {
#                 "candidate": candidate.label,
#                 "label": candidate.label,
#                 "model": candidate.model,
#                 "result": None,
#                 "error": f"{type(e).__name__}: {e}",
#                 "input_tokens": None,
#                 "output_tokens": None,
#                 "cost_usd": None,
#             }


# def _pa_merge_results(
#     existing_results: list[dict],
#     new_results: list[dict],
# ) -> list[dict]:
#     """Merge new results into existing, keyed by label."""
#     merged = {r["label"]: r for r in existing_results}
#     for r in new_results:
#         merged[r["label"]] = r
#     seen = set()
#     ordered = []
#     for r in existing_results:
#         ordered.append(merged[r["label"]])
#         seen.add(r["label"])
#     for r in new_results:
#         if r["label"] not in seen:
#             ordered.append(merged[r["label"]])
#             seen.add(r["label"])
#     return ordered


# # ── Run all buckets ──────────────────────────────────────────────────────────

# _pa_eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
# _pa_eval_dir.mkdir(parents=True, exist_ok=True)

# _pa_total = sum(len(b["tmdb_ids"]) for b in _pa_config["buckets"].values())
# _pa_idx = 0
# _pa_errors = 0

# for bucket_name, bucket in _pa_config["buckets"].items():
#     print(f"\n{'═' * 60}")
#     print(f"Bucket: {bucket_name} ({len(bucket['tmdb_ids'])} movies)")
#     print(f"{'═' * 60}")

#     _bucket_tasks = []
#     _bucket_movie_info = []

#     for tmdb_id in bucket["tmdb_ids"]:
#         if tmdb_id not in _pa_movies:
#             print(f"  SKIP: tmdb_id {tmdb_id} not loaded")
#             continue
#         m = _pa_movies[tmdb_id]
#         movie = m["movie"]
#         title = movie.title_with_year()

#         # 3 candidates per movie, no emotional_observations for any
#         movie_tasks = []
#         for candidate in PLOT_ANALYSIS_CANDIDATES:
#             movie_tasks.append(_pa_generate_one(
#                 movie, m["plot_synopsis"], m["thematic_observations"],
#                 candidate,
#             ))

#         _bucket_tasks.append(movie_tasks)
#         _bucket_movie_info.append((tmdb_id, title))

#     # Fire all tasks for the bucket concurrently (semaphore limits concurrency)
#     all_coros = [coro for movie_tasks in _bucket_tasks for coro in movie_tasks]
#     all_results = await asyncio.gather(*all_coros)

#     # Partition results back to per-movie groups (3 results per movie)
#     results_per_movie = []
#     offset = 0
#     for movie_tasks in _bucket_tasks:
#         n = len(movie_tasks)
#         results_per_movie.append(all_results[offset:offset + n])
#         offset += n

#     # Save per-movie results
#     for (tmdb_id, title), movie_results in zip(_bucket_movie_info, results_per_movie):
#         _pa_idx += 1

#         m = _pa_movies[tmdb_id]
#         user_prompt = build_plot_analysis_user_prompt(
#             m["movie"], m["plot_synopsis"], m["thematic_observations"],
#             None,
#         )

#         successes = sum(1 for r in movie_results if r["result"] is not None)
#         failures = len(movie_results) - successes
#         _pa_errors += failures

#         total_cost = sum(r["cost_usd"] or 0 for r in movie_results)

#         status = "✓" if failures == 0 else f"⚠ {failures} failed"
#         print(f"  [{_pa_idx}/{_pa_total}] {title}: {status}, ${total_cost:.4f}")

#         # Load existing file and merge
#         out_path = _pa_eval_dir / f"plot_analysis_{tmdb_id}.json"
#         if out_path.exists():
#             existing = _json.loads(out_path.read_text())
#             merged = _pa_merge_results(
#                 existing.get("candidate_results", []), list(movie_results),
#             )
#         else:
#             merged = list(movie_results)

#         movie_output = {
#             "tmdb_id": tmdb_id,
#             "title": title,
#             "bucket": bucket_name,
#             "user_prompt": user_prompt,
#             "candidate_results": merged,
#         }
#         out_path.write_text(
#             _json.dumps(movie_output, indent=2, ensure_ascii=False),
#         )

# # ── Summary ──────────────────────────────────────────────────────────────────

# _pa_total_cost = 0
# _pa_total_runs = 0
# for bucket_name, bucket in _pa_config["buckets"].items():
#     for tmdb_id in bucket["tmdb_ids"]:
#         out_path = _pa_eval_dir / f"plot_analysis_{tmdb_id}.json"
#         if out_path.exists():
#             data = _json.loads(out_path.read_text())
#             for r in data["candidate_results"]:
#                 _pa_total_runs += 1
#                 _pa_total_cost += r.get("cost_usd") or 0

# print(f"\n{'═' * 60}")
# print(f"COMPLETE: {_pa_total} movies × 3 runs = {_pa_total_runs} generations")
# print(f"Total cost: ${_pa_total_cost:.4f}")
# print(f"Errors: {_pa_errors}")
# print(f"Results saved to: {_pa_eval_dir}/plot_analysis_*.json")

In [9]:
# ── Viewer Experience (Wave 2) — Round 4: GPO-Only Narrative ─────────────────
# Tests whether generalized_plot_overview alone is a sufficient narrative
# source — strips plot_summary and all raw plot fallbacks so the only
# narrative the model sees is GPO (or nothing if GPO doesn't exist).
#
# Candidates:
#   gpo-only: gpt-5-mini, minimal reasoning, low verbosity,
#             justifications schema, narrative restricted to GPO
#
# Parallelism: all movies launched concurrently (semaphore caps in-flight)

import json as _json
import asyncio
import time
from dataclasses import replace as dc_replace

from movie_ingestion.metadata_generation.generators.viewer_experience import (
    build_viewer_experience_user_prompt,
    generate_viewer_experience,
)
from movie_ingestion.metadata_generation.schemas import (
    ViewerExperienceWithJustificationsOutput,
)
from movie_ingestion.metadata_generation.prompts.viewer_experience import (
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as VE_SYSTEM_PROMPT_JUSTIFICATIONS,
)

# ── Candidate ────────────────────────────────────────────────────────────────
# Single candidate: same model/params as production winner, but narrative
# input restricted to generalized_plot_overview only.

ve_candidates = [
    PlaygroundCandidate(
        label="gpo-only",
        provider=LLMProvider.OPENAI,
        model="gpt-5-mini",
        system_prompt=VE_SYSTEM_PROMPT_JUSTIFICATIONS,
        response_format=ViewerExperienceWithJustificationsOutput,
        kwargs={"reasoning_effort": "minimal", "verbosity": "low"},
    ),
]

# ── Load bucket definitions ──────────────────────────────────────────────────

_ve_buckets_path = project_root / "ingestion_data" / "viewer_experience_eval_buckets.json"
with open(_ve_buckets_path) as _f:
    _ve_buckets = _json.load(_f)

# Collect all tmdb_ids across all buckets, preserving bucket membership
ve_eval_groups: dict[str, list[int]] = {}
ve_all_ids: list[int] = []
for bucket_name, bucket_data in _ve_buckets["buckets"].items():
    ids = [m["tmdb_id"] for m in bucket_data["movies"]]
    ve_eval_groups[bucket_name] = ids
    ve_all_ids.extend(ids)

# Deduplicate while preserving order (a movie shouldn't appear in multiple buckets,
# but guard against it)
ve_all_ids = list(dict.fromkeys(ve_all_ids))
print(f"Loaded {len(ve_eval_groups)} buckets, {len(ve_all_ids)} unique movies total")
for bname, bids in ve_eval_groups.items():
    print(f"  {bname}: {len(bids)} movies")

# ── Build MovieInputData + load Wave 1 metadata for each movie ──────────────

_ve_db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
_ve_db.row_factory = sqlite3.Row

_ve_placeholders = ",".join("?" * len(ve_all_ids))

# TMDB base data
_ve_tmdb_rows = {
    row["tmdb_id"]: row
    for row in _ve_db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({_ve_placeholders})",
        ve_all_ids,
    ).fetchall()
}

# IMDB scraped data
_ve_imdb_rows = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in _ve_db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({_ve_placeholders})",
        ve_all_ids,
    ).fetchall()
}

# Wave 1 generated metadata (plot_events, reception, plot_analysis)
_ve_gen_rows = {
    row["tmdb_id"]: row
    for row in _ve_db.execute(
        f"SELECT tmdb_id, plot_events, reception, plot_analysis "
        f"FROM generated_metadata WHERE tmdb_id IN ({_ve_placeholders})",
        ve_all_ids,
    ).fetchall()
}
_ve_db.close()


def _build_ve_movie(tmdb_id: int) -> MovieInputData | None:
    """Build MovieInputData from tmdb + imdb rows."""
    tmdb = _ve_tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = _ve_imdb_rows.get(tmdb_id, {})
    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )


def _extract_wave1_inputs(tmdb_id: int) -> dict:
    """Extract Wave 1 metadata fields needed by generate_viewer_experience.

    Returns a dict of kwargs ready to spread into the generator call.
    """
    gen_row = _ve_gen_rows.get(tmdb_id)
    if gen_row is None:
        return {}

    inputs = {}

    # plot_events → plot_summary
    pe_raw = gen_row["plot_events"]
    if pe_raw:
        pe = _json.loads(pe_raw)
        inputs["plot_summary"] = pe.get("plot_summary")

    # reception → emotional/craft/thematic observations
    rec_raw = gen_row["reception"]
    if rec_raw:
        rec = _json.loads(rec_raw)
        inputs["emotional_observations"] = rec.get("emotional_observations")
        inputs["craft_observations"] = rec.get("craft_observations")
        inputs["thematic_observations"] = rec.get("thematic_observations")

    # plot_analysis → generalized_plot_overview, genre_signatures, character_arcs
    pa_raw = gen_row["plot_analysis"]
    if pa_raw:
        pa = _json.loads(pa_raw)
        inputs["generalized_plot_overview"] = pa.get("generalized_plot_overview")
        inputs["genre_signatures"] = pa.get("genre_signatures")
        # character_arcs are stored as list of dicts with arc_transformation_label;
        # the generator expects a list of label strings
        raw_arcs = pa.get("character_arcs")
        if raw_arcs and isinstance(raw_arcs, list):
            inputs["character_arcs"] = [
                arc["arc_transformation_label"]
                for arc in raw_arcs
                if isinstance(arc, dict) and "arc_transformation_label" in arc
            ]

    return inputs


# Build MovieInputData and Wave 1 inputs for all evaluation movies
ve_movies: dict[int, MovieInputData] = {}
ve_wave1_inputs: dict[int, dict] = {}
for tid in ve_all_ids:
    m = _build_ve_movie(tid)
    if m is not None:
        ve_movies[tid] = m
        ve_wave1_inputs[tid] = _extract_wave1_inputs(tid)

print(f"\nBuilt {len(ve_movies)} MovieInputData objects with Wave 1 inputs")
print(f"Using {len(ve_candidates)} candidate(s):")
for c in ve_candidates:
    print(f"  - {c.label}")

# ── GPO-only narrative restriction ───────────────────────────────────────────
# Strip plot_summary from wave1 and zero out raw plot fields on the movie
# object so best_plot_fallback() returns None. This forces the narrative
# fallback logic to only see generalized_plot_overview (or nothing).


def _restrict_to_gpo(
    movie: MovieInputData, wave1: dict,
) -> tuple[MovieInputData, dict]:
    """Return modified movie + wave1 with narrative restricted to GPO only."""
    # Remove plot_summary so fallback step 1 sees nothing
    restricted_wave1 = {k: v for k, v in wave1.items() if k != "plot_summary"}

    # Zero out raw plot sources so best_plot_fallback() returns None
    restricted_movie = dc_replace(
        movie,
        plot_synopses=[],
        plot_summaries=[],
        overview="",
    )
    return restricted_movie, restricted_wave1


# ── Generation helper ────────────────────────────────────────────────────────

_VE_CONCURRENCY = asyncio.Semaphore(50)


async def _ve_generate_for_candidate(
    movie: MovieInputData,
    wave1: dict,
    candidate: PlaygroundCandidate,
) -> dict:
    """Generate viewer experience for one (movie, candidate) pair.

    Restricts narrative input to generalized_plot_overview only.
    """
    effective_movie, effective_wave1 = _restrict_to_gpo(movie, wave1)

    async with _VE_CONCURRENCY:
        try:
            result, token_usage = await generate_viewer_experience(
                effective_movie,
                provider=candidate.provider,
                model=candidate.model,
                system_prompt=candidate.system_prompt,
                response_format=candidate.response_format,
                **effective_wave1,
                **candidate.kwargs,
            )
            cost = _compute_cost(
                token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
            )
            cost_str = f"${cost:.6f}" if cost is not None else "n/a"
            print(
                f"  ✓ {candidate.label}: "
                f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
                f"cost={cost_str}"
            )
            return {
                "candidate": candidate.label,
                "model": candidate.model,
                "result": result.model_dump(),
                "input_tokens": token_usage.input_tokens,
                "output_tokens": token_usage.output_tokens,
                "cost_usd": cost,
            }
        except Exception as e:
            print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
            return {
                "candidate": candidate.label,
                "model": candidate.model,
                "result": None,
                "error": f"{type(e).__name__}: {e}",
                "input_tokens": None,
                "output_tokens": None,
                "cost_usd": None,
            }


def _ve_merge_results(
    existing_results: dict[str, dict],
    new_results: dict[str, dict],
) -> dict[str, dict]:
    """Merge new candidate results into existing ones, keyed by label."""
    merged = dict(existing_results)
    merged.update(new_results)
    return merged


# ── Launch all movies in parallel ────────────────────────────────────────────

eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
eval_dir.mkdir(parents=True, exist_ok=True)

# Build ordered list of (tmdb_id, movie, wave1) tuples
_ve_movie_list = [
    (tid, ve_movies[tid], ve_wave1_inputs.get(tid, {}))
    for tid in ve_all_ids
    if tid in ve_movies
]

total_movies = len(_ve_movie_list)
total_tasks = total_movies * len(ve_candidates)
t_start = time.monotonic()

print(f"Launching generation: {total_movies} movies × {len(ve_candidates)} candidate(s) = {total_tasks} tasks")
print(f"Concurrency limit: {_VE_CONCURRENCY._value}\n")

# Build ALL coroutines at once
all_coros = []
all_keys: list[tuple[int, str]] = []  # (tmdb_id, candidate_label)
for tmdb_id, movie, wave1 in _ve_movie_list:
    for candidate in ve_candidates:
        all_keys.append((tmdb_id, candidate.label))
        all_coros.append(
            _ve_generate_for_candidate(movie, wave1, candidate)
        )

all_results = await asyncio.gather(*all_coros)

# Group results by movie
results_per_movie: dict[int, dict[str, dict]] = {}
for (tmdb_id, cand_label), result in zip(all_keys, all_results):
    results_per_movie.setdefault(tmdb_id, {})[cand_label] = result

# Save each movie's results
total_errors = 0
total_cost = 0.0
for tmdb_id, movie, wave1 in _ve_movie_list:
    title = movie.title_with_year()
    # Save the full-input user_prompt as the reference (unpruned)
    user_prompt = build_viewer_experience_user_prompt(movie, **wave1)
    candidate_dict = results_per_movie.get(tmdb_id, {})

    movie_errors = sum(1 for r in candidate_dict.values() if r["result"] is None)
    movie_cost = sum(r["cost_usd"] or 0 for r in candidate_dict.values())
    total_errors += movie_errors
    total_cost += movie_cost

    status = "ok" if movie_errors == 0 else f"{movie_errors} failed"
    print(f"  {title}: {status}, ${movie_cost:.4f}")

    # Load existing file and merge (preserves results from prior rounds)
    out_path = eval_dir / f"viewer_experience_{tmdb_id}.json"
    if out_path.exists():
        existing = _json.loads(out_path.read_text())
        merged = _ve_merge_results(
            existing.get("candidate_results", {}), candidate_dict,
        )
    else:
        merged = candidate_dict

    movie_output = {
        "tmdb_id": tmdb_id,
        "title": title,
        "user_prompt": user_prompt,
        "candidate_results": merged,
    }
    out_path.write_text(_json.dumps(movie_output, indent=2, ensure_ascii=False))

elapsed = time.monotonic() - t_start
print(f"\nDone: {total_movies} movies × {len(ve_candidates)} candidate(s)")
print(f"Total cost: ${total_cost:.4f} | Errors: {total_errors}")
print(f"Elapsed: {elapsed:.1f}s ({total_tasks / elapsed:.1f} gen/sec)")

Loaded 7 buckets, 50 unique movies total
  gold_standard: 8 movies
  floor_plot_summary: 8 movies
  raw_fallback_standalone: 8 movies
  raw_fallback_with_observations: 6 movies
  generalized_plot_overview: 8 movies
  obs_standalone_with_context: 6 movies
  obs_standalone_minimal_context: 6 movies

Built 50 MovieInputData objects with Wave 1 inputs
Using 1 candidate(s):
  - gpo-only
Launching generation: 50 movies × 1 candidate(s) = 50 tasks
Concurrency limit: 50

  ✓ gpo-only: 3693 in / 638 out, cost=$0.002199
  ✓ gpo-only: 3725 in / 762 out, cost=$0.002455
  ✓ gpo-only: 3766 in / 769 out, cost=$0.002479
  ✓ gpo-only: 3774 in / 686 out, cost=$0.002315
  ✓ gpo-only: 3686 in / 753 out, cost=$0.002427
  ✓ gpo-only: 3757 in / 703 out, cost=$0.002345
  ✓ gpo-only: 3596 in / 684 out, cost=$0.002267
  ✓ gpo-only: 3675 in / 684 out, cost=$0.002287
  ✓ gpo-only: 3729 in / 816 out, cost=$0.002564
  ✓ gpo-only: 3681 in / 740 out, cost=$0.002400
  ✓ gpo-only: 3789 in / 777 out, cost=$0.002501
  ✓ 

In [10]:
# ── Narrative Techniques (Wave 2) ─────────────────────────────────────────────
# Candidates: gpt-5-mini low, gpt-5.4-nano medium
# Rate target: 120 generations/sec (60 movies × 2 candidates)
# Inputs: MovieInputData + Wave 1 metadata (plot_events → plot_summary,
#         reception → craft_observations)

import json as _json
import asyncio
import time
from movie_ingestion.metadata_generation.generators.narrative_techniques import (
    build_narrative_techniques_user_prompt,
    generate_narrative_techniques,
)

# ── Candidates ────────────────────────────────────────────────────────────────

nt_candidates = [
    PlaygroundCandidate(
        "gpt-5-mini-low", LLMProvider.OPENAI, "gpt-5-mini",
        {"reasoning_effort": "low"},
    ),
    PlaygroundCandidate(
        "gpt-5.4-nano-medium", LLMProvider.OPENAI, "gpt-5.4-nano",
        {"reasoning_effort": "medium"},
    ),
]

# ── Load bucket definitions ──────────────────────────────────────────────────

_nt_buckets_path = project_root / "ingestion_data" / "narrative_techniques_eval_buckets.json"
with open(_nt_buckets_path) as _f:
    _nt_buckets = _json.load(_f)

# Collect all tmdb_ids across all buckets, preserving bucket membership
nt_eval_groups: dict[str, list[int]] = {}
nt_all_ids: list[int] = []
for bucket_name, bucket_data in _nt_buckets["buckets"].items():
    ids = [m["tmdb_id"] for m in bucket_data["movies"]]
    nt_eval_groups[bucket_name] = ids
    nt_all_ids.extend(ids)

# Deduplicate while preserving order
nt_all_ids = list(dict.fromkeys(nt_all_ids))
print(f"Loaded {len(nt_eval_groups)} buckets, {len(nt_all_ids)} unique movies total")
for bname, bids in nt_eval_groups.items():
    print(f"  {bname}: {len(bids)} movies")

# ── Build MovieInputData + load Wave 1 metadata for each movie ──────────────

_nt_db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
_nt_db.row_factory = sqlite3.Row

_nt_placeholders = ",".join("?" * len(nt_all_ids))

# TMDB base data
_nt_tmdb_rows = {
    row["tmdb_id"]: row
    for row in _nt_db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({_nt_placeholders})",
        nt_all_ids,
    ).fetchall()
}

# IMDB scraped data
_nt_imdb_rows = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in _nt_db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({_nt_placeholders})",
        nt_all_ids,
    ).fetchall()
}

# Wave 1 generated metadata (plot_events, reception)
_nt_gen_rows = {
    row["tmdb_id"]: row
    for row in _nt_db.execute(
        f"SELECT tmdb_id, plot_events, reception "
        f"FROM generated_metadata WHERE tmdb_id IN ({_nt_placeholders})",
        nt_all_ids,
    ).fetchall()
}
_nt_db.close()


def _build_nt_movie(tmdb_id: int) -> MovieInputData | None:
    """Build MovieInputData from tmdb + imdb rows."""
    tmdb = _nt_tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = _nt_imdb_rows.get(tmdb_id, {})
    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )


def _extract_nt_wave1_inputs(tmdb_id: int) -> dict:
    """Extract Wave 1 metadata fields needed by generate_narrative_techniques.

    Returns a dict of kwargs: plot_summary and craft_observations.
    """
    gen_row = _nt_gen_rows.get(tmdb_id)
    if gen_row is None:
        return {}

    inputs = {}

    # plot_events → plot_summary
    pe_raw = gen_row["plot_events"]
    if pe_raw:
        pe = _json.loads(pe_raw)
        inputs["plot_summary"] = pe.get("plot_summary")

    # reception → craft_observations
    rec_raw = gen_row["reception"]
    if rec_raw:
        rec = _json.loads(rec_raw)
        inputs["craft_observations"] = rec.get("craft_observations")

    return inputs


# Build MovieInputData and Wave 1 inputs for all evaluation movies
nt_movies: dict[int, MovieInputData] = {}
nt_wave1_inputs: dict[int, dict] = {}
for tid in nt_all_ids:
    m = _build_nt_movie(tid)
    if m is not None:
        nt_movies[tid] = m
        nt_wave1_inputs[tid] = _extract_nt_wave1_inputs(tid)

print(f"\nBuilt {len(nt_movies)} MovieInputData objects with Wave 1 inputs")
print(f"Using {len(nt_candidates)} candidates:")
for c in nt_candidates:
    print(f"  - {c.label} ({c.model}, reasoning_effort={c.kwargs.get('reasoning_effort', 'default')})")

# ── Rate-limited generation ──────────────────────────────────────────────────
# Target: 120 generations/sec → 60 movies/sec (2 candidates each).
# Use a semaphore to cap in-flight requests and a token-bucket style
# delay to stay under the rate ceiling.

_NT_MAX_CONCURRENT = 60  # max in-flight generation calls
_NT_RATE_LIMIT = 120     # generations per second
_nt_semaphore = asyncio.Semaphore(_NT_MAX_CONCURRENT)
_nt_rate_interval = 1.0 / _NT_RATE_LIMIT  # seconds between allowed starts


async def _nt_generate_for_candidate(
    movie: MovieInputData,
    wave1: dict,
    candidate: PlaygroundCandidate,
    rate_lock: asyncio.Lock,
    last_start: list[float],
) -> dict:
    """Generate narrative techniques for one (movie, candidate) pair with rate limiting."""
    async with _nt_semaphore:
        # Token-bucket: ensure minimum interval between generation starts
        async with rate_lock:
            now = time.monotonic()
            elapsed = now - last_start[0]
            if elapsed < _nt_rate_interval:
                await asyncio.sleep(_nt_rate_interval - elapsed)
            last_start[0] = time.monotonic()

        try:
            result, token_usage = await generate_narrative_techniques(
                movie,
                provider=candidate.provider,
                model=candidate.model,
                **wave1,
                **candidate.kwargs,
            )
            cost = _compute_cost(
                token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
            )
            cost_str = f"${cost:.6f}" if cost is not None else "n/a"
            print(
                f"  ✓ {candidate.label}: "
                f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
                f"cost={cost_str}"
            )
            return {
                "candidate": candidate.label,
                "model": candidate.model,
                "result": result.model_dump(),
                "input_tokens": token_usage.input_tokens,
                "output_tokens": token_usage.output_tokens,
                "cost_usd": cost,
            }
        except Exception as e:
            print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
            return {
                "candidate": candidate.label,
                "model": candidate.model,
                "result": None,
                "error": f"{type(e).__name__}: {e}",
                "input_tokens": None,
                "output_tokens": None,
                "cost_usd": None,
            }


# ── Generate ALL movies × candidates in parallel ─────────────────────────────

eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
eval_dir.mkdir(parents=True, exist_ok=True)

nt_rate_lock = asyncio.Lock()
nt_last_start = [0.0]

# Build flat list of coroutines and launch all at once
_nt_coros = []
_nt_task_keys: list[tuple[int, str]] = []  # (tmdb_id, candidate_label)

for tmdb_id, movie in nt_movies.items():
    wave1 = nt_wave1_inputs.get(tmdb_id, {})
    for candidate in nt_candidates:
        _nt_task_keys.append((tmdb_id, candidate.label))
        _nt_coros.append(
            _nt_generate_for_candidate(movie, wave1, candidate, nt_rate_lock, nt_last_start)
        )

print(f"Launching {len(_nt_coros)} generation tasks "
      f"({len(nt_movies)} movies × {len(nt_candidates)} candidates)...")
t_start = time.monotonic()

_nt_all_results = await asyncio.gather(*_nt_coros)

elapsed = time.monotonic() - t_start
print(f"\nAll generations complete in {elapsed:.1f}s "
      f"({len(_nt_coros) / elapsed:.1f} generations/sec)")

# ── Group results by movie and save ──────────────────────────────────────────

_nt_per_movie: dict[int, dict[str, dict]] = {}
for (tmdb_id, cand_label), result in zip(_nt_task_keys, _nt_all_results):
    _nt_per_movie.setdefault(tmdb_id, {})[cand_label] = result

# Save per-movie, preserving bucket grouping for logging
nt_total_errors = 0
nt_total_cost = 0.0
for bucket_name, bucket_ids in nt_eval_groups.items():
    bucket_movies = [(tid, nt_movies[tid]) for tid in bucket_ids if tid in nt_movies]
    if not bucket_movies:
        continue
    print(f"\n{'='*60}")
    print(f"Bucket: {bucket_name} ({len(bucket_movies)} movies)")
    print(f"{'='*60}")

    for movie_idx, (tmdb_id, movie) in enumerate(bucket_movies, 1):
        title = movie.title_with_year()
        wave1 = nt_wave1_inputs.get(tmdb_id, {})
        user_prompt = build_narrative_techniques_user_prompt(movie, **wave1)
        candidate_dict = _nt_per_movie.get(tmdb_id, {})

        # Tally errors and cost
        movie_errors = sum(1 for r in candidate_dict.values() if r["result"] is None)
        movie_cost = sum(r["cost_usd"] or 0 for r in candidate_dict.values())
        nt_total_errors += movie_errors
        nt_total_cost += movie_cost

        status = "ok" if movie_errors == 0 else f"{movie_errors} failed"
        print(f"  [{movie_idx}/{len(bucket_movies)}] {title}: {status}, ${movie_cost:.4f}")

        movie_output = {
            "tmdb_id": tmdb_id,
            "title": title,
            "user_prompt": user_prompt,
            "candidate_results": candidate_dict,
        }

        out_path = eval_dir / f"narrative_techniques_{tmdb_id}.json"
        out_path.write_text(_json.dumps(movie_output, indent=2, ensure_ascii=False))

print(f"\nDone: {len(nt_movies)} movies × {len(nt_candidates)} candidates")
print(f"Total cost: ${nt_total_cost:.4f} | Errors: {nt_total_errors}")

Loaded 7 buckets, 56 unique movies total
  gold_standard: 8 movies
  floor_tier1: 8 movies
  raw_fallback_standalone: 8 movies
  craft_standalone_rich: 8 movies
  craft_standalone_floor: 8 movies
  combined_path: 8 movies
  craft_threshold_edge: 8 movies

Built 56 MovieInputData objects with Wave 1 inputs
Using 2 candidates:
  - gpt-5-mini-low (gpt-5-mini, reasoning_effort=low)
  - gpt-5.4-nano-medium (gpt-5.4-nano, reasoning_effort=medium)
Launching 112 generation tasks (56 movies × 2 candidates)...
  ✓ gpt-5.4-nano-medium: 2880 in / 432 out, cost=$0.001116
  ✓ gpt-5.4-nano-medium: 2930 in / 512 out, cost=$0.001226
  ✓ gpt-5.4-nano-medium: 2973 in / 544 out, cost=$0.001275
  ✓ gpt-5.4-nano-medium: 2844 in / 547 out, cost=$0.001253
  ✓ gpt-5.4-nano-medium: 2940 in / 591 out, cost=$0.001327
  ✓ gpt-5.4-nano-medium: 3184 in / 633 out, cost=$0.001428
  ✓ gpt-5.4-nano-medium: 2932 in / 662 out, cost=$0.001414
  ✓ gpt-5.4-nano-medium: 2966 in / 644 out, cost=$0.001398
  ✓ gpt-5.4-nano-mediu

CancelledError: 